# Volume 批次增量入库

本地仅采集上传；本 notebook 用 Spark 计算处理 `_ready` 下的完成清单，不调用 SQL Warehouse。
检查所有未成功批次（含前几天）；没有新数据就正常结束。同店同日使用最新发布批次。
**仅通过同一个 Job 运行，最大并发 1。不要同时运行旧版入库 notebook 或本地 --ingest。**
处理记录：`workspace.default.bronze_price_ingestion_runs`。SUCCEEDED 表示发布的完整门店文件已处理，不代表整天所有门店都已采齐。


In [ ]:
dbutils.widgets.text("volume_path", "/Volumes/workspace/default/price_tracker")
volume=dbutils.widgets.get("volume_path").rstrip("/")


## 1. 批次协议与最新版本选择

In [ ]:
"""Immutable publication contract; shared by local publisher and cloud notebook."""
import hashlib
import json
import re
from datetime import date


def validate_batch(batch, volume):
    if batch.get('protocol_version') != 1 or batch.get('mode') != 'production':
        raise ValueError('Unsupported batch protocol')
    if not re.fullmatch(r'production_\d{8}T\d{6}Z_[0-9a-f]{8}',batch.get('run_id','')):
        raise ValueError('Invalid run ID')
    day=date.fromisoformat(batch['collected_date'])
    if day<date(2026,10,27):raise ValueError('Before official collection start')
    seen=set()
    for record in batch['files']:
        key=(record['source'],record['store_id'])
        if key in seen:raise ValueError('Duplicate store batch')
        seen.add(key)
        if record['source'] not in ('woolworths','newworld','paknsave'):raise ValueError('Unknown source')
        if not re.fullmatch(r'[A-Za-z0-9-]+',record['store_id']):raise ValueError('Invalid store ID')
        path=record['remote_file']
        if not path.startswith(volume+'/prices_'+record['source']+'_') or '/' in path[len(volume)+1:] or not path.endswith('.csv'):
            raise ValueError('Unexpected file path')
        if not isinstance(record['rows'],int) or record['rows']<=0:raise ValueError('Empty batch')
        if any(record['category_counts'].get(c,0)<=0 for c in ('Egg','Milk','Meat')):raise ValueError('Missing category')
    if not seen:raise ValueError('No complete store files')
    return batch


def digest(batch):
    return hashlib.sha256(json.dumps(batch,sort_keys=True,separators=(',',':')).encode()).hexdigest()


def latest_files(batches):
    """Choose newest published run per source/store/day, including processed runs."""
    latest={}
    for batch in batches:
        for record in batch['files']:
            key=(record['source'],record['store_id'],batch['collected_date'])
            if key not in latest or batch['run_id']>latest[key][0]:
                latest[key]=(batch['run_id'],record)
    return latest


## 2. 已有 Bronze → Silver 转换计划（复用统一 SQL 规则）

In [ ]:
plan = {"refresh": [], "views": []}


### refresh · 1

In [ ]:
plan['refresh'].append(r"""
MERGE INTO workspace.default.silver_dim_store t
USING (SELECT DISTINCT concat(source, ':', store_id) AS store_key, source, store_id,
source AS brand_id, 'NZ' AS country_code FROM workspace.default.bronze_prices) s
ON t.store_key = s.store_key
WHEN NOT MATCHED THEN INSERT (store_key,source,store_id,brand_id,country_code,location_verified)
VALUES (s.store_key,s.source,s.store_id,s.brand_id,s.country_code,false)
""")


### refresh · 2

In [ ]:
plan['refresh'].append(r"""
MERGE INTO workspace.default.silver_dim_date t USING (
 SELECT DISTINCT cast(date_format(collected_date,'yyyyMMdd') AS INT) date_id,collected_date calendar_date,
 year(collected_date) year,quarter(collected_date) quarter,month(collected_date) month,day(collected_date) day,
 weekday(collected_date)+1 day_of_week,weekday(collected_date)>=5 is_weekend FROM workspace.default.bronze_prices
) s ON t.date_id=s.date_id WHEN NOT MATCHED THEN INSERT *
""")


### refresh · 3

In [ ]:
plan['refresh'].append(r"""
CREATE OR REPLACE VIEW workspace.default.silver_v_prices_clean AS
WITH parsed AS (
 SELECT *,
 lower(regexp_replace(trim(coalesce(total_quant,'')), '\\s+', '')) raw_spec_key,
 lower(trim(coalesce(unit,''))) raw_unit_key,
 nullif(regexp_extract(lower(original_name), '(\\d+(?:\\.\\d+)?\\s*(?:ml|kg|pack|pk|g|l|ea))(?:\\s+(?:bottle|carton))?\\s*$',1),'') name_spec,
 lower(original_name) RLIKE '(min\\s+order|\\d+(?:\\.\\d+)?\\s*[-–]\\s*\\d+(?:\\.\\d+)?\\s*(kg|g))' variable_weight
 FROM workspace.default.bronze_prices WHERE collected_date >= DATE '2026-10-27'
), normalized AS (
 SELECT *,
 CASE WHEN variable_weight THEN NULL
 WHEN raw_spec_key RLIKE '^(\\d+(\\.\\d+)?(ml|l|g|kg|pk|pack|ea)|kg|ea)$'
 THEN replace(raw_spec_key,'pack','pk')
 ELSE replace(regexp_replace(name_spec,'\\s+',''),'pack','pk') END clean_spec,
 CASE WHEN raw_unit_key RLIKE '^kg\\s*\\$' THEN 'kg'
 WHEN raw_unit_key IN ('kg','l','ea','100g','100ml') THEN raw_unit_key END clean_unit
 FROM parsed
)
SELECT * EXCEPT(total_quant,unit,raw_spec_key,raw_unit_key,name_spec,variable_weight,clean_spec,clean_unit),
 total_quant raw_specification,unit raw_reported_unit,
 clean_spec total_quant,clean_unit unit,
 CASE WHEN variable_weight THEN 'variable_weight' WHEN clean_spec IS NULL THEN 'unknown'
 WHEN raw_spec_key IN ('','na') THEN 'parsed_from_name' ELSE 'reported' END specification_quality_status,
 CASE WHEN clean_unit IS NULL THEN 'unknown' ELSE 'ok' END unit_quality_status,
 CASE WHEN clean_unit IN ('kg','100g') THEN 'kg' WHEN clean_unit IN ('l','100ml') THEN 'l'
 WHEN clean_unit='ea' THEN 'ea' END normalized_unit,
 CASE WHEN clean_unit IS NOT NULL AND try_cast(unit_price AS DECIMAL(16,6))>0
 THEN try_cast(unit_price AS DECIMAL(16,6))*CASE WHEN clean_unit IN ('100g','100ml') THEN 10 ELSE 1 END END normalized_unit_price,
 CASE WHEN clean_unit IS NOT NULL AND try_cast(member_unit_price AS DECIMAL(16,6))>0
 THEN try_cast(member_unit_price AS DECIMAL(16,6))*CASE WHEN clean_unit IN ('100g','100ml') THEN 10 ELSE 1 END END normalized_member_unit_price
FROM normalized
""")


### refresh · 4

In [ ]:
plan['refresh'].append(r"""
MERGE INTO workspace.default.silver_dim_product_source t
USING (WITH alias_map AS (
 SELECT a.source, lower(regexp_replace(trim(coalesce(a.original_name,'')), '\\s+', ' ')) name_key,
 replace(lower(regexp_replace(trim(coalesce(a.original_specification,'')), '\\s+', '')), 'pack', 'pk') spec_key, p.category_id,
 min(a.product_id) product_id
 FROM workspace.default.silver_dim_product_alias a
 JOIN workspace.default.silver_dim_product p ON a.product_id=p.product_id
 GROUP BY a.source,lower(regexp_replace(trim(coalesce(a.original_name,'')), '\\s+', ' ')),replace(lower(regexp_replace(trim(coalesce(a.original_specification,'')), '\\s+', '')), 'pack', 'pk'),p.category_id
 HAVING count(DISTINCT a.product_id)=1
), candidates AS (
 SELECT b.source,b.source_product_id,min(a.product_id) product_id,
 max(b.original_name) original_name,max(b.total_quant) original_specification,
 max(try_cast(b.category_id AS INT)) category_id,max(b.product_url) product_url
 FROM workspace.default.silver_v_prices_clean b LEFT JOIN alias_map a
 ON b.source=a.source AND lower(regexp_replace(trim(coalesce(b.original_name,'')), '\\s+', ' '))=a.name_key
 AND replace(lower(regexp_replace(trim(coalesce(b.total_quant,'')), '\\s+', '')), 'pack', 'pk')=a.spec_key AND try_cast(b.category_id AS INT)=a.category_id
 WHERE b.source_product_id IS NOT NULL AND trim(b.source_product_id)<>''
 GROUP BY b.source,b.source_product_id
 HAVING count(DISTINCT a.product_id)<=1
) SELECT *,CASE WHEN product_id IS NOT NULL THEN 'exact_reference' ELSE 'pending' END mapping_status,
 'Exact name/spec/category reference match; no fuzzy matching' mapping_note FROM candidates) s
ON t.source=s.source AND t.source_product_id=s.source_product_id
WHEN MATCHED AND t.mapping_status='pending' AND s.product_id IS NOT NULL THEN UPDATE SET
 t.product_id=s.product_id,t.mapping_status=s.mapping_status,t.mapping_note=s.mapping_note
WHEN NOT MATCHED THEN INSERT *
""")


### refresh · 5

In [ ]:
plan['refresh'].append(r"""
MERGE INTO workspace.default.silver_dim_product_source t
USING (SELECT 'woolworths' source,v.*,max(b.product_url) product_url FROM
 (SELECT * FROM VALUES ('52242','0003','Farmer Brown Eggs Cage Free Barn Size 7 12 Pack','12pk',1),('774152','0011','Woodland Eggs Free Range Size 6 10 Pack','10pk',1),('774153','0014','Woodland Eggs Free Range Size 7 10 Pack','10pk',1),('546293','0019','Meadow Fresh Milk Standard Original 1L Bottle','1l',2),('282793','0020','Meadow Fresh Milk Standard Original 2L Bottle','2l',2),('701971','0021','Meadow Fresh Milk Standard Original 3L Bottle','3l',2),('546321','0022','Meadow Fresh Calci-Trim Milk Trim Calci-Trim 1L Bottle','1l',2),('282729','0023','Meadow Fresh Calci-Trim Milk Trim Calci-Trim 2L Bottle','2l',2) AS v(source_product_id,product_id,original_name,original_specification,category_id)) v
 JOIN workspace.default.silver_v_prices_clean b ON b.source='woolworths' AND b.source_product_id=v.source_product_id
 AND b.original_name=v.original_name AND b.total_quant=v.original_specification AND try_cast(b.category_id AS INT)=v.category_id
 GROUP BY ALL) s ON t.source=s.source AND t.source_product_id=s.source_product_id
WHEN MATCHED AND t.mapping_status IN ('pending','exact_reference') THEN UPDATE SET
 t.product_id=s.product_id,t.mapping_status='approved',t.original_specification=s.original_specification,
 t.mapping_note='Reviewed exact source SKU, brand, product line, egg grade/farming method or fresh bottle volume; 2026-10-27'
WHEN NOT MATCHED THEN INSERT (source,source_product_id,product_id,original_name,original_specification,category_id,product_url,mapping_status,mapping_note)
VALUES (s.source,s.source_product_id,s.product_id,s.original_name,s.original_specification,s.category_id,s.product_url,'approved','Reviewed exact source card identity; 2026-10-27')
""")


### refresh · 6

In [ ]:
plan['refresh'].append(r"""
UPDATE workspace.default.silver_dim_product_source SET
 product_id=NULL,mapping_status='pending_review',
 mapping_note='Two distinct source SKUs share Original Fresh Milk 1L title. Review bottle/carton/product identity before comparison; 2026-10-27'
WHERE source IN ('newworld','paknsave') AND source_product_id IN ('product-5026254-EA-000','product-5026255-EA-000')
AND mapping_status IN ('pending','exact_reference')
""")


### refresh · 7

In [ ]:
plan['refresh'].append(r"""
UPDATE workspace.default.silver_dim_product SET comparison_unit='l' WHERE category_id=2 AND is_analysis_selected=true AND comparison_unit<>'l'
""")


### refresh · 8

In [ ]:
plan['refresh'].append(r"""
INSERT OVERWRITE TABLE workspace.default.silver_fact_product_price
(observation_id,date_id,collected_date,store_key,product_id,category_id,source,store_id,original_name,original_specification,total_price,reported_unit_price,reported_unit,is_promo,product_mapping_status,price_quality_status,source_file,ingested_at,source_product_id,product_url,member_price,member_unit_price,was_price,store_verification_status,raw_specification,raw_reported_unit,specification_quality_status,unit_quality_status,normalized_unit,normalized_unit_price,normalized_member_unit_price)
WITH alias_map AS (
 SELECT a.source, lower(regexp_replace(trim(coalesce(a.original_name,'')), '\\s+', ' ')) name_key,
 replace(lower(regexp_replace(trim(coalesce(a.original_specification,'')), '\\s+', '')), 'pack', 'pk') spec_key, p.category_id,
 min(a.product_id) product_id
 FROM workspace.default.silver_dim_product_alias a
 JOIN workspace.default.silver_dim_product p ON a.product_id=p.product_id
 GROUP BY a.source,lower(regexp_replace(trim(coalesce(a.original_name,'')), '\\s+', ' ')),replace(lower(regexp_replace(trim(coalesce(a.original_specification,'')), '\\s+', '')), 'pack', 'pk'),p.category_id
 HAVING count(DISTINCT a.product_id)=1
), ranked AS (
 SELECT *,row_number() OVER (
 PARTITION BY source,store_id,collected_date,category_id,
 CASE WHEN nullif(source_product_id,'') IS NOT NULL THEN source_product_id ELSE concat_ws('||',original_name,coalesce(total_quant,'')) END
 ORDER BY ingested_at DESC,source_file DESC,total_price DESC,unit_price DESC,is_promo DESC
 ) rn FROM workspace.default.silver_v_prices_clean WHERE collected_date >= DATE '2026-10-27'
), enriched AS (
 SELECT b.*,p.product_id mapped_product_id,
 coalesce(p.category_id,a.category_id,c.category_id) mapped_category_id,
 CASE WHEN p.product_id IS NOT NULL THEN 'source_id' WHEN s.mapping_status='pending_review' THEN 'pending_review' ELSE 'unmapped' END mapping_method,
 try_cast(b.total_price AS DECIMAL(12,4)) price_value,
 try_cast(b.unit_price AS DECIMAL(12,4)) unit_price_value
 FROM ranked b
 LEFT JOIN workspace.default.silver_dim_product_source s
 ON b.source=s.source AND b.source_product_id=s.source_product_id
 LEFT JOIN workspace.default.silver_dim_product p ON s.product_id=p.product_id AND s.mapping_status IN ('approved','exact_reference') AND p.category_id=try_cast(b.category_id AS INT) AND replace(lower(regexp_replace(trim(coalesce(p.specification,'')), '\\s+', '')), 'pack', 'pk')=replace(lower(regexp_replace(trim(coalesce(b.total_quant,'')), '\\s+', '')), 'pack', 'pk')
 LEFT JOIN alias_map a ON b.source=a.source AND lower(regexp_replace(trim(coalesce(b.original_name,'')), '\\s+', ' '))=a.name_key
 AND replace(lower(regexp_replace(trim(coalesce(b.total_quant,'')), '\\s+', '')), 'pack', 'pk')=a.spec_key AND try_cast(b.category_id AS INT)=a.category_id
 LEFT JOIN workspace.default.silver_dim_category c ON try_cast(b.category_id AS INT)=c.category_id
 WHERE b.rn=1
)
SELECT sha2(to_json(named_struct('source',source,'store_id',store_id,'date',collected_date,'category',category_id,
 'identity',CASE WHEN nullif(source_product_id,'') IS NOT NULL THEN source_product_id ELSE concat_ws('||',original_name,coalesce(total_quant,'')) END)),256),
 cast(date_format(collected_date,'yyyyMMdd') AS INT),collected_date,concat(source,':',store_id),mapped_product_id,mapped_category_id,
 source,store_id,original_name,total_quant,
 CASE WHEN price_value>=0 THEN price_value END,CASE WHEN unit_price_value>=0 THEN unit_price_value END,unit,
 CASE WHEN is_promo='1' THEN true WHEN is_promo='0' THEN false END,mapping_method,
 CASE WHEN price_value IS NULL THEN 'missing_or_invalid_price' WHEN price_value<0 THEN 'negative_price' WHEN price_value=0 THEN 'zero_price' ELSE 'ok' END,
 source_file,ingested_at,source_product_id,product_url,
 CASE WHEN try_cast(member_price AS DECIMAL(12,4))>=0 THEN try_cast(member_price AS DECIMAL(12,4)) END,
 CASE WHEN try_cast(member_unit_price AS DECIMAL(12,4))>=0 THEN try_cast(member_unit_price AS DECIMAL(12,4)) END,
 CASE WHEN try_cast(was_price AS DECIMAL(12,4))>=0 THEN try_cast(was_price AS DECIMAL(12,4)) END,
 CASE WHEN try_cast(store_verified AS BOOLEAN)=true THEN 'verified' ELSE 'legacy_unverified' END,
raw_specification,raw_reported_unit,specification_quality_status,unit_quality_status,normalized_unit,normalized_unit_price,normalized_member_unit_price
FROM enriched
""")


### views · 1

In [ ]:
plan['views'].append(r"""
CREATE OR REPLACE VIEW workspace.default.gold_v_analysis_products AS
SELECT p.*,c.category_name,
 (SELECT count(DISTINCT s.source) FROM workspace.default.silver_dim_product_source s WHERE s.product_id=p.product_id AND s.mapping_status IN ('approved','exact_reference')) source_id_brand_coverage
FROM workspace.default.silver_dim_product p JOIN workspace.default.silver_dim_category c ON p.category_id=c.category_id
WHERE p.is_analysis_selected=true
""")


### views · 2

In [ ]:
plan['views'].append(r"""
CREATE OR REPLACE VIEW workspace.default.gold_v_analysis_prices AS
WITH candidates AS (
 SELECT f.*,p.product_name,p.specification standard_specification,p.comparison_unit,p.analysis_rank,
 p.comparison_ready product_comparison_ready,
 count(*) OVER (PARTITION BY f.collected_date,f.store_key,f.product_id) mapped_skus_in_store
 FROM workspace.default.silver_fact_product_price f
 JOIN workspace.default.silver_dim_product p ON f.product_id=p.product_id WHERE p.is_analysis_selected=true
)
SELECT * EXCEPT(product_comparison_ready),
 coalesce(product_comparison_ready AND store_verification_status='verified' AND price_quality_status='ok'
 AND unit_quality_status='ok' AND normalized_unit_price>0 AND mapped_skus_in_store=1
 AND specification_quality_status IN ('reported','parsed_from_name')
 AND product_mapping_status IN ('source_id','name_spec_exact'),false) comparison_ready
FROM candidates
""")


### views · 3

In [ ]:
plan['views'].append(r"""
CREATE OR REPLACE VIEW workspace.default.gold_v_unmapped_products AS
SELECT source,source_product_id,category_id,original_name,original_specification,count(*) observations,
 min(collected_date) first_seen,max(collected_date) last_seen
FROM workspace.default.silver_fact_product_price WHERE product_id IS NULL
GROUP BY source,source_product_id,category_id,original_name,original_specification
""")


### views · 4

In [ ]:
plan['views'].append(r"""
CREATE OR REPLACE VIEW workspace.default.silver_v_product_mapping_queue AS
SELECT f.source,f.source_product_id,f.category_id,max(f.original_name) original_name,
 max(f.original_specification) normalized_specification,max(f.raw_specification) raw_specification,
 max(f.product_url) product_url,max(s.mapping_status) registry_status,max(s.mapping_note) review_note,
 count(*) observations,count(DISTINCT f.store_key) stores,min(f.collected_date) first_seen,max(f.collected_date) last_seen
FROM workspace.default.silver_fact_product_price f
LEFT JOIN workspace.default.silver_dim_product_source s ON f.source=s.source AND f.source_product_id=s.source_product_id
WHERE f.product_id IS NULL GROUP BY f.source,f.source_product_id,f.category_id
""")


### views · 5

In [ ]:
plan['views'].append(r"""
CREATE OR REPLACE VIEW workspace.default.silver_v_price_quality AS
SELECT collected_date,source,category_id,count(*) observations,count(DISTINCT store_key) stores,
 count_if(product_id IS NULL) unmapped,count_if(price_quality_status<>'ok') invalid_price,
 count_if(unit_quality_status<>'ok') unknown_unit,count_if(specification_quality_status='unknown') unknown_specification,
 count_if(specification_quality_status='variable_weight') variable_weight
FROM workspace.default.silver_fact_product_price GROUP BY collected_date,source,category_id
""")


## 3. 增量入库、处理记录与验证

In [ ]:
"""Notebook runtime. Spark and dbutils are supplied by Databricks, not a SQL Warehouse.

Only the scheduled Job may write these tables; max_concurrent_runs must be 1.
The generated notebook embeds this module, batch_protocol and the SQL plan.
"""
import json
from datetime import datetime
from zoneinfo import ZoneInfo


def process_batches(spark, dbutils, volume, plan):
    from pyspark.sql import functions as F
    ledger='workspace.default.bronze_price_ingestion_runs'
    def quote(value):return "'"+str(value).replace('\\','\\\\').replace("'","''")+"'"
    def sql(query):return spark.sql(query).collect()
    sql(f'''CREATE TABLE IF NOT EXISTS {ledger} (
        run_id STRING, manifest_hash STRING, collected_date DATE, status STRING,
        row_count BIGINT, store_count BIGINT, error_message STRING, updated_at TIMESTAMP
    ) USING DELTA''')
    dbutils.fs.mkdirs(volume+'/_ready')
    batches=[]
    today=datetime.now(ZoneInfo('Pacific/Auckland')).date().isoformat()
    for file in dbutils.fs.ls(volume+'/_ready'):
        if not file.name.endswith('.json'):continue
        with open(file.path.removeprefix('dbfs:'),encoding='utf-8') as stream:batch=json.load(stream)
        validate_batch(batch,volume)
        if batch['collected_date']>today:raise ValueError('Future-dated batch')
        batches.append(batch)
    if len({b['run_id'] for b in batches})!=len(batches):raise ValueError('Duplicate run IDs')
    known={r.run_id:r.asDict() for r in sql(f'SELECT * FROM {ledger}')}
    for batch in batches:
        prior=known.get(batch['run_id'])
        if prior and prior['manifest_hash']!=digest(batch):raise ValueError('Published manifest changed')
    latest=latest_files(batches)
    pending=[b for b in sorted(batches,key=lambda b:b['run_id'])
             if known.get(b['run_id'],{}).get('status') not in ('SUCCEEDED','SUPERSEDED')]
    if not pending:
        print('NO_NEW_BATCHES: no unprocessed published batches, including previous dates.')
        return {'status':'no_new_batches','processed':0}

    def record(batch,status,rows=0,stores=0,error=''):
        sql(f'''MERGE INTO {ledger} t USING (SELECT
            {quote(batch['run_id'])} run_id,{quote(digest(batch))} manifest_hash,
            DATE {quote(batch['collected_date'])} collected_date,{quote(status)} status,
            CAST({rows} AS BIGINT) row_count,CAST({stores} AS BIGINT) store_count,
            {quote(error[:4000])} error_message,current_timestamp() updated_at) s
            ON t.run_id=s.run_id WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *''')

    fields=['product_id','store_id','category_id','total_price','total_quant','unit','unit_price',
            'original_name','collected_date','is_promo','source','source_product_id','product_url',
            'member_price','member_unit_price','was_price','store_verified']
    schema=','.join(k+' STRING' for k in fields)
    applied=[]
    try:
        for batch in pending:
            run_id=batch['run_id']
            chosen=[r for r in batch['files'] if latest[(r['source'],r['store_id'],batch['collected_date'])][0]==run_id]
            if not chosen:
                record(batch,'SUPERSEDED')
                continue
            record(batch,'RUNNING')
            print('PROCESS',run_id,'stores',len(chosen),'of',batch['expected_stores'])
            frames=[]
            expected={}
            predicates=[]
            for item in chosen:
                frame=(spark.read.format('csv').schema(schema).option('header','true')
                       .option('multiLine','true').option('escape','"').option('mode','FAILFAST')
                       .load(item['remote_file'])
                       .withColumn('collected_date',F.to_date('collected_date'))
                       .withColumn('source_file',F.lit(item['remote_file']))
                       .withColumn('ingested_at',F.current_timestamp()))
                # Validate each file's identity, not merely totals across the union.
                summary=frame.groupBy('source','store_id','collected_date').count().collect()
                key=(item['source'],item['store_id'],batch['collected_date'])
                if len(summary)!=1 or (summary[0]['source'],summary[0]['store_id'],str(summary[0]['collected_date']))!=key or summary[0]['count']!=item['rows']:
                    raise ValueError('File identity/row count mismatch: '+item['remote_file'])
                frames.append(frame);expected[key]=item['rows']
                predicates.append(f"(source={quote(key[0])} AND store_id={quote(key[1])} AND collected_date=DATE {quote(key[2])})")
            stage=frames[0]
            for frame in frames[1:]:stage=stage.unionByName(frame)
            view='_price_batch_stage'
            stage.createOrReplaceTempView(view)
            checks=sql(f'''SELECT source,store_id,count(DISTINCT category_id) categories,
                count_if(coalesce(try_cast(store_verified AS BOOLEAN),false)=false) unverified,
                count_if(nullif(source_product_id,'') IS NULL) missing_sku,
                count(*)-count(DISTINCT struct(category_id,source_product_id)) duplicates
                FROM {view} GROUP BY 1,2''')
            if any(r.categories!=3 or r.unverified or r.missing_sku or r.duplicates for r in checks):
                raise ValueError('Category/store/SKU validation failed')
            predicate=' OR '.join(predicates)
            columns=','.join('`'+c+'`' for c in spark.table('workspace.default.bronze_prices').columns)
            # Each store/day replacement is idempotent. Ledger commits are separate;
            # a crash before SUCCEEDED causes safe replacement and transformation retry.
            sql(f'INSERT INTO workspace.default.bronze_prices REPLACE WHERE {predicate} SELECT {columns} FROM {view}')
            observed=sql(f'''SELECT source,store_id,cast(collected_date AS STRING) day,count(*) n
                FROM workspace.default.bronze_prices WHERE {predicate} GROUP BY 1,2,3''')
            if {(r.source,r.store_id,r.day):r.n for r in observed}!=expected:raise ValueError('Bronze verification failed')
            applied.append((batch,sum(expected.values()),len(chosen),predicate))
            record(batch,'BRONZE_WRITTEN',sum(expected.values()),len(chosen))
        if applied:
            # Once per scheduled execution, not once per store or batch.
            for index,statement in enumerate(plan['refresh']+plan['views'],1):
                print('TRANSFORM',index)
                sql(statement)
            integrity=sql('''SELECT count(*)-count(DISTINCT observation_id) duplicates,
                count_if(collected_date < DATE '2026-10-27') historical
                FROM workspace.default.silver_fact_product_price''')[0]
            if integrity.duplicates or integrity.historical:raise ValueError('Silver integrity failure')
            for batch,rows,stores,predicate in applied:
                counts=sql(f'''SELECT count(*) rows,count(DISTINCT store_key) stores,
                    count_if(coalesce(store_verification_status,'')<>'verified') unverified
                    FROM workspace.default.silver_fact_product_price WHERE {predicate}''')[0]
                if counts.rows!=rows or counts.stores!=stores or counts.unverified:
                    raise ValueError('Silver batch count/store verification failed')
            for batch,rows,stores,_ in applied:record(batch,'SUCCEEDED',rows,stores)
        print('COMPLETE',len(applied),'batches; partial collections remain explicitly marked in each ready manifest.')
        return {'status':'complete','processed':len(applied)}
    except Exception as error:
        # Preserve successful earlier records; RUNNING/BRONZE_WRITTEN/FAILED all retry.
        for batch in pending:
            rows=sql(f'SELECT status FROM {ledger} WHERE run_id={quote(batch["run_id"])}')
            if rows and rows[0].status in ('RUNNING','BRONZE_WRITTEN'):
                record(batch,'FAILED',error=str(error))
        raise


## 4. 执行

In [ ]:
result=process_batches(spark,dbutils,volume,plan)
result['ledger']=[r.asDict() for r in spark.sql("SELECT run_id,cast(collected_date AS STRING) collected_date,status,row_count,store_count FROM workspace.default.bronze_price_ingestion_runs ORDER BY collected_date,run_id").collect()]
dbutils.fs.mkdirs(volume+'/_processing')
dbutils.fs.put(volume+'/_processing/latest_result.json',json.dumps(result,ensure_ascii=False,indent=2),True)
dbutils.notebook.exit(json.dumps(result))
